# Quantización alternativa MLP27K Montmeló para FINN

Este notebook prueba una idea más prudente:

1. Partimos del checkpoint bueno `mlp_27k.ckpt`.
2. Creamos una MLP Brevitas con la misma arquitectura `576 → 32 → 256 → 2`.
3. Probamos varias versiones cuantizadas:
   - entrada cuantizada, capas cuantizadas, salida sin cuantizar;
   - pesos cuantizados a 8 bits y 4 bits;
   - con y sin cuantización de activaciones.
4. Comparamos cada versión contra el `data.csv`.
5. Solo si alguna versión mantiene buen error, se exporta a ONNX para FINN.

La idea es **no pasar a FINN una red mala**. Primero tiene que parecerse al CSV.


In [1]:
from pathlib import Path
import os
import csv
import math
import shutil
import numpy as np
from PIL import Image

import torch
import torch.nn as nn

import brevitas.nn as qnn
import brevitas.onnx as bo
from brevitas.quant import Int8Bias

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.core.datatype import DataType
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    GiveUniqueNodeNames,
    GiveReadableTensorNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)

print("Torch:", torch.__version__)
print("Working dir:", Path.cwd())


Torch: 1.7.1
Working dir: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example


## 1. Rutas

In [2]:
CKPT_PATH = Path("mlp_27k.ckpt")

# Dataset: se buscan varias rutas posibles
candidate_dirs = [
    Path("output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/Orig/DATA_12_16/counter_montmelo_holonomic"),
]

DATASET_DIR = None
for d in candidate_dirs:
    if (d / "data.csv").exists():
        DATASET_DIR = d
        break

if not CKPT_PATH.exists():
    raise FileNotFoundError("No encuentro mlp_27k.ckpt en la carpeta del notebook")

if DATASET_DIR is None:
    raise FileNotFoundError("No encuentro el data.csv del dataset")

CSV_PATH = DATASET_DIR / "data.csv"

print("Checkpoint:", CKPT_PATH.resolve())
print("Dataset:", DATASET_DIR)
print("CSV:", CSV_PATH)


Checkpoint: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/mlp_27k.ckpt
Dataset: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic
CSV: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic/data.csv


## 2. Funciones comunes

In [3]:
VMIN = -0.6
VMAX = 13.0
WMIN = -math.pi
WMAX = math.pi

def denorm_v_01(vnorm):
    return (VMAX - VMIN) * vnorm + VMIN

def denorm_w_01(wnorm):
    return (WMAX - WMIN) * wnorm + WMIN

def norm_v_01(v):
    return (v - VMIN) / (VMAX - VMIN)

def norm_w_01(w):
    return (w - WMIN) / (WMAX - WMIN)

def image_to_flat_01(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    arr = arr / 255.0
    return torch.tensor(arr, dtype=torch.float32).unsqueeze(0)

def image_to_flat_uint8(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    return arr.reshape(1, 576).astype(np.float32)

def load_rows(n_samples=2000):
    rows = []
    with open(CSV_PATH) as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_path = DATASET_DIR / row["image_name"]
            if not img_path.exists():
                continue
            rows.append({
                "image_name": row["image_name"],
                "img_path": img_path,
                "v": float(row["v"]),
                "w": float(row["w"]),
            })
            if len(rows) >= n_samples:
                break
    return rows

rows = load_rows(2000)
print("Frames cargados:", len(rows))


Frames cargados: 2000


## 3. Modelo float de referencia

In [4]:
class FloatMLP27K(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(576, 32)
        self.fx = nn.Linear(32, 256)
        self.predictor = nn.Linear(256, 2)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fx(x))
        x = self.predictor(x)
        return x

def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ["state_dict", "model_state_dict", "net", "model"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return obj.state_dict()

def clean_state_dict(sd):
    out = {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        nk = k
        changed = True
        prefixes = ["model.", "base_model.", "module.", "net."]
        while changed:
            changed = False
            for pref in prefixes:
                if nk.startswith(pref):
                    nk = nk[len(pref):]
                    changed = True
        nk = nk.replace("predictor.0.", "predictor.")
        out[nk] = v
    return out

obj = torch.load(str(CKPT_PATH), map_location="cpu")
sd = clean_state_dict(extract_state_dict(obj))

print("Tensores encontrados:")
for k, v in sd.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape))

float_model = FloatMLP27K()
float_model.load_state_dict(sd, strict=True)
float_model.eval()

print("Modelo float cargado correctamente")


Tensores encontrados:
fc1.weight (32, 576)
fc1.bias (32,)
fx.weight (256, 32)
fx.bias (256,)
predictor.weight (2, 256)
predictor.bias (2,)
Modelo float cargado correctamente


## 4. Evaluación de modelos PyTorch

In [6]:
def eval_pytorch_model(model, rows, name="model", print_first=10):
    model.eval()
    err_v, err_w = [], []
    pred_v, pred_w = [], []

    with torch.no_grad():
        for i, r in enumerate(rows):
            x = image_to_flat_01(r["img_path"])
            y = model(x).view(-1).detach().cpu().numpy()

            vnorm = float(y[0])
            wnorm = float(y[1])

            v_pred = denorm_v_01(vnorm)
            w_pred = denorm_w_01(wnorm)

            v_csv = r["v"]
            w_csv = r["w"]

            err_v.append(v_pred - v_csv)
            err_w.append(w_pred - w_csv)
            pred_v.append(v_pred)
            pred_w.append(w_pred)

            if i < print_first:
                raw_v_csv = round(norm_v_01(v_csv) * 255)
                raw_w_csv = round(norm_w_01(w_csv) * 255)
                raw_v_pred = round(vnorm * 255)
                raw_w_pred = round(wnorm * 255)
                print(
                    f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} "
                    f"raw_csv={raw_v_csv},{raw_w_csv} | "
                    f"{name} v={v_pred:.4f} w={w_pred:.4f} raw={raw_v_pred},{raw_w_pred}"
                )

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    res = {
        "name": name,
        "frames": len(rows),
        "mae_v": float(np.mean(np.abs(err_v))),
        "mae_w": float(np.mean(np.abs(err_w))),
        "std_v": float(pred_v.std()),
        "std_w": float(pred_w.std()),
        "v_min": float(pred_v.min()),
        "v_max": float(pred_v.max()),
        "w_min": float(pred_w.min()),
        "w_max": float(pred_w.max()),
    }

    print()
    print(name)
    print("Frames:", res["frames"])
    print("MAE_v =", res["mae_v"])
    print("MAE_w =", res["mae_w"])
    print("v pred min/max/std =", res["v_min"], res["v_max"], res["std_v"])
    print("w pred min/max/std =", res["w_min"], res["w_max"], res["std_w"])

    return res

float_res = eval_pytorch_model(float_model, rows, name="FloatMLP27K")
float_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | FloatMLP27K v=4.8432 w=-0.1681 raw=102,121
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121

FloatMLP27K
Frames: 2000
MAE_v = 0.6021009925648568
MAE_w = 0.1371983721378483

{'name': 'FloatMLP27K',
 'frames': 2000,
 'mae_v': 0.6021009925648568,
 'mae_w': 0.13719837213784833,
 'std_v': 1.2614186193435533,
 'std_w': 0.6229009507532546,
 'v_min': -0.237543985247612,
 'v_max': 6.210170078277588,
 'w_min': -1.5784582724599736,
 'w_max': 1.6049311443956347}

## 5. Modelos cuantizados alternativos

Probamos varias opciones. La anterior iba mal porque la salida quedaba muy saturada. Aquí probamos primero sin cuantizar la salida y con una entrada mejor definida en `[0,1]`.


In [7]:
class QuantMLP27K_NoQuantOut(nn.Module):
    def __init__(self, weight_bit_width=8, act_bit_width=8, use_act_quant=True):
        super().__init__()
        self.use_act_quant = use_act_quant

        self.quant_in = qnn.QuantIdentity(
            bit_width=act_bit_width,
            return_quant_tensor=True,
            min_val=0.0,
            max_val=1.0
        )

        self.fc1 = qnn.QuantLinear(
            576, 32,
            bias=True,
            weight_bit_width=weight_bit_width,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu1 = qnn.QuantReLU(
            bit_width=act_bit_width,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            32, 256,
            bias=True,
            weight_bit_width=weight_bit_width,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu2 = qnn.QuantReLU(
            bit_width=act_bit_width,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            256, 2,
            bias=True,
            weight_bit_width=weight_bit_width,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.quant_in(x)
        x = self.fc1(x)
        if self.use_act_quant:
            x = self.relu1(x)
        else:
            x = torch.relu(x.value if hasattr(x, "value") else x)
        x = self.fx(x)
        if self.use_act_quant:
            x = self.relu2(x)
        else:
            x = torch.relu(x.value if hasattr(x, "value") else x)
        x = self.predictor(x)
        if hasattr(x, "value"):
            x = x.value
        return x

def load_float_weights_into_quant(qmodel, sd):
    qsd = qmodel.state_dict()
    loaded = []
    for k, v in sd.items():
        if k in qsd and tuple(qsd[k].shape) == tuple(v.shape):
            qsd[k] = v
            loaded.append(k)
    qmodel.load_state_dict(qsd, strict=False)
    return loaded

variants = []
for wbits in [8, 6, 4]:
    for abits in [8, 6]:
        for use_act in [True, False]:
            name = f"Q_NoOut_w{wbits}_a{abits}_act{int(use_act)}"
            qm = QuantMLP27K_NoQuantOut(weight_bit_width=wbits, act_bit_width=abits, use_act_quant=use_act)
            loaded = load_float_weights_into_quant(qm, sd)
            print(name, "loaded:", loaded)
            variants.append((name, qm))


Q_NoOut_w8_a8_act1 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w8_a8_act0 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w8_a6_act1 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w8_a6_act0 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w6_a8_act1 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w6_a8_act0 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w6_a6_act1 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w6_a6_act0 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.bias', 'predictor.weight', 'predictor.bias']
Q_NoOut_w4_a8_act1 loaded: ['fc1.weight', 'fc1.bias', 'fx.weight', 'fx.b

## 6. Comparar variantes cuantizadas contra CSV

In [9]:
results = []

for name, qm in variants:
    print("=" * 100)
    print("Probando:", name)

    try:
        res = eval_pytorch_model(qm, rows, name=name, print_first=3)
        results.append(res)
    except Exception as e:
        print("ERROR en variante:", name)
        print(repr(e))
        print("Se salta esta variante y seguimos.")

print()
print("=" * 100)
print("RANKING")
print("=" * 100)

results_sorted = sorted(results, key=lambda r: r["mae_v"] + r["mae_w"])

for i, r in enumerate(results_sorted, 1):
    print(
        f"{i:02d}. {r['name']:25s} "
        f"MAE_v={r['mae_v']:.4f} MAE_w={r['mae_w']:.4f} "
        f"std_v={r['std_v']:.4f} std_w={r['std_w']:.4f} "
        f"v=[{r['v_min']:.3f},{r['v_max']:.3f}] "
        f"w=[{r['w_min']:.3f},{r['w_max']:.3f}]"
    )


Probando: Q_NoOut_w8_a8_act1
1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | Q_NoOut_w8_a8_act1 v=0.2234 w=-3.0283 raw=15,5
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_NoOut_w8_a8_act1 v=0.2221 w=-3.0350 raw=15,4
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_NoOut_w8_a8_act1 v=0.2221 w=-3.0350 raw=15,4

Q_NoOut_w8_a8_act1
Frames: 2000
MAE_v = 4.87282105890708
MAE_w = 2.970398933008432
v pred min/max/std = -1.182587018609047 2.1119387984275817 0.29788787341876766
w pred min/max/std = -3.3086942371271917 -1.8483008852879206 0.204133246322969
Probando: Q_NoOut_w8_a8_act0
ERROR en variante: Q_NoOut_w8_a8_act0
RuntimeError('Input scale required')
Se salta esta variante y seguimos.
Probando: Q_NoOut_w8_a6_act1
1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | Q_NoOut_w8_a6_act1 v=0.6693 w=-2.8199 raw=24,13
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_NoOut_w8_a6_act1 v=0.6693 w=-2.8199 raw=24,13
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_NoOut_w8_a6_act1 v=0.6693 w=

## 7. Elegir mejor variante

Si el mejor resultado sigue siendo malo, hay que hacer fine-tuning cuantizado.  
Como regla práctica:

- Si `MAE_v` queda cerca de `0.6` y `MAE_w` cerca de `0.13`, podemos intentar exportar.
- Si queda mucho peor, no tiene sentido meterlo a FINN todavía.


In [ ]:
BEST_NAME = results_sorted[0]["name"]
print("Mejor variante:", BEST_NAME)

best_model = None
for name, qm in variants:
    if name == BEST_NAME:
        best_model = qm
        break

print(results_sorted[0])


## 8. Exportar mejor variante a ONNX FINN

Ejecuta esta celda **solo si el ranking tiene un error razonable**.


In [ ]:
EXPORT_QUANT_ONNX = Path("mlp27k_quantized_noout_best.onnx")

# Descomenta si quieres exportar:
bo.export_finn_onnx(
    best_model,
    input_shape=(1, 576),
    export_path=str(EXPORT_QUANT_ONNX)
)

print("Exportado:", EXPORT_QUANT_ONNX)
!ls -lh mlp27k_quantized_noout_best.onnx


## 9. Tidy-up básico del ONNX exportado

In [ ]:
MODEL_FOR_FINN = Path("model_for_finn_quantized.onnx")
MODEL_WORK = Path("model.onnx")

model = ModelWrapper(str(EXPORT_QUANT_ONNX))
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model = model.transform(RemoveUnusedTensors())

# La entrada del modelo cuantizado ahora sigue siendo 0-1 si no hemos metido preproc.
# Para FINN con bytes del testbench, luego habrá que decidir si añadimos ToTensor /255.
model.save(str(MODEL_FOR_FINN))
shutil.copyfile(MODEL_FOR_FINN, MODEL_WORK)

print("Guardado:", MODEL_FOR_FINN)
print("Copiado como:", MODEL_WORK)


## 10. Probar ONNX cuantizado contra CSV

Probamos el ONNX exportado antes de hacer build. En este caso la entrada va en `0-1` porque no hemos metido todavía preprocesado `/255`.


In [ ]:
model = ModelWrapper(str(MODEL_FOR_FINN))
model = model.transform(InferShapes())

input_name = model.graph.input[0].name
output_name = model.graph.output[0].name

print("Input:", input_name, model.get_tensor_shape(input_name), model.get_tensor_datatype(input_name))
print("Output:", output_name, model.get_tensor_shape(output_name))

def eval_onnx_01(model, rows, name="ONNX"):
    err_v, err_w = [], []
    pred_v, pred_w = [], []
    input_name = model.graph.input[0].name
    output_name = model.graph.output[0].name

    for i, r in enumerate(rows):
        x = image_to_flat_01(r["img_path"]).numpy().astype(np.float32)
        y_dict = execute_onnx(model, {input_name: x})
        y = y_dict[output_name].reshape(-1)

        vnorm = float(y[0])
        wnorm = float(y[1])
        v_pred = denorm_v_01(vnorm)
        w_pred = denorm_w_01(wnorm)

        v_csv = r["v"]
        w_csv = r["w"]

        err_v.append(v_pred - v_csv)
        err_w.append(w_pred - w_csv)
        pred_v.append(v_pred)
        pred_w.append(w_pred)

        if i < 10:
            print(f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} | {name} v={v_pred:.4f} w={w_pred:.4f}")

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    print()
    print(name)
    print("MAE_v =", np.mean(np.abs(err_v)))
    print("MAE_w =", np.mean(np.abs(err_w)))
    print("v pred min/max/std =", pred_v.min(), pred_v.max(), pred_v.std())
    print("w pred min/max/std =", pred_w.min(), pred_w.max(), pred_w.std())

eval_onnx_01(model, rows, name="ONNX_quantized_noout")


## 11. Si sigue mal: fine-tuning cuantizado

Si todas las variantes cuantizadas siguen dando mal, esta es la siguiente idea: entrenar unas épocas la red cuantizada usando el CSV.  
Esta celda prepara un dataset sencillo y hace fine-tuning con L1Loss sobre las salidas normalizadas `[0,1]`.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class MontmeloDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        x = image_to_flat_01(r["img_path"]).view(-1)
        y = torch.tensor([norm_v_01(r["v"]), norm_w_01(r["w"])], dtype=torch.float32)
        return x, y

dataset = MontmeloDataset(rows)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

ft_model = best_model
ft_model.train()

optimizer = torch.optim.Adam(ft_model.parameters(), lr=1e-4, weight_decay=1e-6)
loss_fn = nn.L1Loss()

EPOCHS = 10

for epoch in range(EPOCHS):
    total = 0.0
    n = 0
    for x, y in loader:
        optimizer.zero_grad()
        pred = ft_model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * x.size(0)
        n += x.size(0)
    print(f"Epoch {epoch+1}/{EPOCHS} loss={total/n:.6f}")

ft_model.eval()
ft_res = eval_pytorch_model(ft_model, rows, name="FineTunedQuant", print_first=10)
ft_res


## 12. Exportar modelo cuantizado fine-tuned

In [ ]:
FT_ONNX = Path("mlp27k_quantized_finetuned.onnx")

bo.export_finn_onnx(
    ft_model,
    input_shape=(1, 576),
    export_path=str(FT_ONNX)
)

print("Exportado:", FT_ONNX)
!ls -lh mlp27k_quantized_finetuned.onnx


## 13. Preparar FINN_BUILD_DIR

In [ ]:
os.environ["FINN_BUILD_DIR"] = "/tmp/FINN_build_mlp27k_quant"
Path(os.environ["FINN_BUILD_DIR"]).mkdir(parents=True, exist_ok=True)
print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])


## 14. Intentar build_dataflow

Ejecuta esta parte solo cuando el ONNX cuantizado tenga resultados aceptables contra el CSV.


In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

FINAL_OUTPUT_DIR = "output_final_mlp27k_quant"

if Path(FINAL_OUTPUT_DIR).exists():
    shutil.rmtree(FINAL_OUTPUT_DIR)
    print("Borrado output anterior:", FINAL_OUTPUT_DIR)

# Usamos el ONNX fine-tuned como model.onnx.
# Si quieres usar otro, cambia esta línea.
shutil.copyfile(FT_ONNX, "model.onnx")

cfg = build.DataflowBuildConfig(
    output_dir=FINAL_OUTPUT_DIR,
    folding_config_file=None,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    stitched_ip_gen_dcp=True,
    split_large_fifos=True,
    synth_clk_period_ns=10.0,
    board=None,
    fpga_part="xc7z020clg400-1",
    default_mem_mode="const",
    shell_flow_type=None,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ]
)

print("Lanzando FINN con model.onnx")
try:
    build.build_dataflow_cfg("model.onnx", cfg)
    print("FINN build terminado")
except Exception as e:
    print("FINN build ha fallado.")
    print("Error:", repr(e))


## 15. Debug si falla FINN

In [ ]:
!tail -100 output_final_mlp27k_quant/build_dataflow.log 2>/dev/null || true


In [ ]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

onnx_files = sorted(Path("/tmp/FINN_build_mlp27k_quant").rglob("*.onnx"))
print("ONNX intermedios:", len(onnx_files))
for i, f in enumerate(onnx_files):
    print(i, f)

if onnx_files:
    last_model = onnx_files[-1]
    print("\nInspeccionando:", last_model)
    m = ModelWrapper(str(last_model))
    for node in m.graph.node:
        print(node.name, node.op_type)
